## HDB Resale Flat Prices - ETL Pipeline
Notebook runs the whole pipeline end to end from grabbing the raw files to creating the tables the Data Science team needs.

Architecture (Medallion Architecture)

| **Layer** | **Output group** | **What it contains**                                                                        |
|-----------|------------------|---------------------------------------------------------------------------------------------|
| Bronze    | Raw              | Files queried and downloaded from data.gov.sg (source) exactly as received. Never modified. |
| Silver    | Cleaned          | Combined and later validated rows with remaining lease and resale-price anomaly flags.      |
| Silver    | Quarantined      | Rows that failed the rejection rules, each with the appropriate rejection reason.           |
| Gold      | Transformed      | Cleaned rows with a unique identifier based on the requirements.                            |
| Gold      | Hashed           | Same rows as with the identifier but it is replaced by a keyed hash (HMAC-SHA256).          |

General contents
- Setup
- Bronze layer: Raw data Ingestion
- Silver layer: Combination, Validation, Calculation of remaining lease and resale price anomalies
- Gold layer: Resale identifier and subsequent de-identification
- ETL Lifecycle profiling
- Deliverables Inventory

How to - Run
1. In your repository terminal, run `pip install -r requirements.txt` (Implementation was done on Python 3.13, but all functions should work on 3.11 and later)
2. In the repository terminal, run `copy .env.example .env`, then place a real secret in `HASH_PEPPER` (this can be achieved by using the following command in the terminal `python -c "import secrets; print(secrets.token_hex(32))"`, and then pasting the result in `HASH_PEPPER` where the `CHANGE_ME` is.
NOTE: Hashes generated will be different from the hashes in the commited files which is the expected behavior due to the method chosen to adhere to requirements. Production behavior and decision making will be discussed in the relevant sections.
3. Ensure that the intended configurations are set in `src/config.yaml`.
NOTE: All paths, URLs, thresholds and rule configurations come from `src/config.yaml`.
4. Run all cells from top to bottom. Ingestion will skip files that already exists to ensure that re-runs are safe.

## 1. Setup
### Import required libraries, load config, config helpers and relevant variables

In [ ]:
import logging
import sys
from pathlib import Path

import polars as pl
from IPython.display import display

# Find the repository root and make src importable
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"config"/"config.yaml").exists())
sys.path.insert(0, str(REPO_ROOT/"src"))

from src.config import load_config, get_hash_secret
from src.logging_setup import setup_logging
from src.bronze.ingest import run_ingestion_pipeline
from src.silver.combine import run_combine, write_combined_df, summarize_ingestion_sizes
from src.silver.profiling import generate_ydata_profile_reference_dataset, profile_stage, generate_ydata_profile
from src.silver.validation import (
    build_authoritative_dataset_value_set, run_validation, quarantine_reason_counts, summarize_rejections, write_validation_outputs
)
from src.silver.calc_remaining_lease import calculate_remaining_lease
from src.silver.anomaly_resale_prices import flag_resale_price_anomalies, summarize_anomalies_report, split_anomalies
from src.gold.identifier import (
    add_resale_identifier, summarize_identifier_collisions, write_transformed_identifier_df
)
from src.gold.deidentify import deidentify_identifier, write_hashed_output_df

# Ensures that polars shows every column in outputs
pl.Config.set_tbl_cols(-1)

# Sets how many rows polars will show in outputs
pl.Config.set_tbl_rows(30)

# Reads config and .env
cfg = load_config()

setup_logging(cfg)

# Check whether hash secret was set correctly
get_hash_secret(cfg=cfg)
print("HASH_PEPPER set and long enough.")

In [ ]:
# Key configurations for the run (all from config/config.yaml) - Please ensure they are set as expected
display(pl.DataFrame({
    "Settings": [
        "collection_id",
        "scope (inclusive)",
        "authoritative month",
        "validated_categorical_columns",
        "lease term/ as of data"
    ],
    "Value": [
        cfg.require("source.collection_id"),
        "{start} to {end}".format(start=cfg.require("scope.start_year_month"), end=cfg.require("scope.end_year_month")),
        "{month}".format(month=cfg.require("reference.authoritative_dataset_month")),
        ", ".join(cfg.require("validation.categorical_columns")),
        "{full_term} years / {as_of_date}".format(
            full_term=cfg.require("calc_remaining_lease.lease_full_term_years"),
            as_of_date=cfg.require("calc_remaining_lease.lease_calc_period_end_date"))
    ]
}, schema={"Settings":pl.String, "Value": pl.String}, strict=False))

## 2. Bronze Layer
### 2.1 Raw data Ingestion
Ingest raw data and output bronze file group.
The collection's child datasets are discovered from the data.gov.sg collection metadata. Then, each file is downloaded to `data_storage/bronze/raw` as is with no modifications at all.

* Downloads retry with backoff on network errors (429 and 5xx responses). Rate limit is honored with a delay configured in config.yaml.
* Files are written to a temporary `.part` file and renamed when complete, to avoid failed downloads leaving a truncated file.
* Existing files are skipped, to ensure runs are idempotent.

In [ ]:
result = run_ingestion_pipeline(cfg=cfg)

# Shows which datasets were downloaded/skipped/failed
print({k: len(v) for k, v in result.items()})

assert not result["failed"],  "Datasets failed to download: {result}".format(result=result['failed'])

In [ ]:
# Raw output files and their sizes (in mb)
raw_dir = cfg.path("bronze_raw")
display(pl.DataFrame([{
    "file": p.name,
    "size_mb": round(p.stat().st_size / 1e6, 2)} for p in sorted(raw_dir.glob("*.csv"))
]))

## 3. Silver Layer
### 3.1 Combination into main dataset

- Every file is read as **strings** (no type inference), so values remain exactly as received (no modifications requirement).
- The file containing the authoritative month (**configured in config.yaml but 2012-01 as per requirements**) defines the authoritative schema (columns and order). Columns that exist only in the later files (such as `remaining_lease` from analysis), are left out of the main dataset and remain intact unchanged in Bronze.
- A `source_file` column saves where each row has come from.
- Nothing is filtered or de-duplicated yet here.

In [ ]:
# Write set to False to show combined dataframe write path
combined_df, frames = run_combine(
    cfg=cfg,
    write=False
)
combined_path = write_combined_df(
    cfg=cfg,
    combined_df=combined_df,
)

# Showcases per-file and combined sizes: evidence for engine choice below
display(summarize_ingestion_sizes(
    cfg=cfg,
    frames=frames,
    combined_df=combined_df,
    parquet_path=combined_path,
))

In [ ]:
# Check: No rows are added or lost during combination
assert combined_df.height == sum(df.height for df in frames.values()), "Row count drift has occurred during combination"

display(pl.DataFrame({
    "column": combined_df.columns,
}))

display("Combined data row count: " + str(combined_df.height))

display(combined_df.head(5))

#### **Engine choice for data operations**
The decision was originally between pySpark or Polars. Using pySpark already would simplify integrating the code into the targeted cloud infrastructure (AWS Glue), however, the size table above and the row count show that the full collection is small enough to process in memory on device. Spark would add a JVM dependency and start-up cost without any realistic benefit at the moment at this size.
Spark + Glue is the scale path and will be described in Part 2 Architecture.


### 3.2 Profiling the authoritative dataset (As per requirements: 2012-01)
The January 2012 rows define what "valid" values are for categorical columns. Therefore, they are profiled first with [ydata-profiling](https://github.com/ydataai/ydata-profiling), an open-source data profiling framework as required.
As ydata doesn't natively support Polars, the small authoritative slice is converted to pandas (where the numeric columns are cast within that profiling copy only so that the pipeline remains untouched at this point).
The HTML and JSON reports are written to `reports/profiling`, and the findings are written in `reports/data_profiling_report.md`.

In [ ]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in generate_ydata_profile
# Takes combined_df and filters down to reference set parameters - 2012-01 for month
reference_report_paths = generate_ydata_profile_reference_dataset(
    cfg=cfg,
    combined_df=combined_df,
)
print(reference_report_paths)

Added a profiling step for the entire data to gather more insights before further steps.

In [ ]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in generate_ydata_profile
silver_combined_stage_name = cfg.require("outputs.silver_combined_filename")
report_paths = generate_ydata_profile(
    cfg=cfg,
    df=combined_df,
    profile_name=silver_combined_stage_name
)
print(report_paths)

In [ ]:
# Lightweight, polars-native profiling (rows, nulls, distinct values per column), reused at every stage during design and implementation
stage_profiles = [profile_stage(df=combined_df, stage="silver_combined")]
display(stage_profiles[0])

### 3.3 Validation and quarantine
Determined rules are evaluated together,so a row that breaks several is quarantined once with every reason listed in `reject_reason`.

| **Rule**                                                                                        | **Reject reason**   |
|-------------------------------------------------------------------------------------------------|---------------------|
| Required column is null or blank                                                                | `MISSING_REQUIRED`  |
| Price, area or lease year not numeric, below its minimum, or lease starts after the sale year   | `INVALID_NUMERIC`   |
| Repeat of an earlier row (all columns except `source_file`; the first occurence is kept         | `DUPLICATE`         |
| town, flat_type, flat_model or storey_range not found in the 2012-01 data (nulls fail here too) | `INVALID_<COLUMN>`  |
| Month outside 2012-01 to 2012-12 (Inclusive)                                                    | `OUT_OF_SCOPE_DATE` |

Since as per the requirements, matching against the authoritative set is **strict**, exact value with no case folding or whitespace trimming is implemented (can be configured in config.yaml). The reference sets are bult from the full combined data, before any row is removed.

In [ ]:
# Allowed values per validated column based on 2012-01 rows only
authoritative_value_sets = build_authoritative_dataset_value_set(
    cfg=cfg,
    combined_df=combined_df,
)
for column, values in authoritative_value_sets.items():
    print("{cols}: {len_values} allowed values -> {values}\n".format(
        cols=column,
        len_values=len(values),
        values=", ".join(str(v) for v in values),
    ))

In [ ]:
# Apply data validation rules
validated_df, quarantined_df = run_validation(
    cfg=cfg,
    combined_df=combined_df
)
print("Validated: {validated_row_count} rows | Quarantined: {quarantined_row_count} rows".format(
    validated_row_count=validated_df.height,
    quarantined_row_count=quarantined_df.height,
))

display(validated_df.head(10))

In [ ]:
display(quarantine_reason_counts(
    cfg=cfg,
    quarantined_df=quarantined_df,
))
display(summarize_rejections(
    cfg=cfg,
    quarantined_df=quarantined_df,
    authoritative_value_sets=authoritative_value_sets,
))
display(quarantined_df.head(10))

### 3.4 Calculate remaining lease as of 2027-01-01
The data only gives the lease start **year**, so the lease is assumed to start in January of that year and run 99 years. Remaining lease is measured from 2027-01-01, rounded down to whole months and split into years and months.
Since the remaining lease is measured from 2027-01-01 (January) and the start month for each lease commence date is also january, the months part is always 0 here (both of which can be configured in config.yaml).

Additionally, expired leases are set to 0 instead of showing negative values.

In [ ]:
remaining_lease_df = calculate_remaining_lease(
    cfg=cfg,
    df=validated_df,
)

data_lease_commence_date_col = cfg.require("scope.data_lease_commence_date_col")
remaining_years_col = cfg.require("calc_remaining_lease.remaining_lease_years_col")
remaining_months_col = cfg.require("calc_remaining_lease.remaining_lease_months_col")
remaining_text_col = cfg.require("calc_remaining_lease.remaining_lease_text_col")

display(
    remaining_lease_df
    .group_by(
        data_lease_commence_date_col,
        remaining_years_col,
        remaining_months_col,
        remaining_text_col,
    ).agg(
        pl.len()
    )
    .rename({"len":"entry_count"})
    .unique()
    .sort(data_lease_commence_date_col).head(10)
)

### 3.5 Detection of potentially anomalous resale prices
Rows deemed to be anomalous are **flagged** but **not removed**, because an unusual resale price could still be a real sale and the requirements didn't indicate the necessity to quarantine these anomalous values.
The following two heuristics are used:
- **Peer-group outliers**: price is compared with other flats of the same town, flat type and year using the IQR rule (configurable). Groups that are too small fall back to flat type and year so thresholds stay stable.
- **Resale price per sqm outside set bounds**: bounds are configured in config.yaml. Present to catch errors that peer statistics may miss, for example if a whole grouping is wrong (albeit this is unlikely).

In [ ]:
anomaly_df = flag_resale_price_anomalies(
    cfg=cfg,
    df=remaining_lease_df,
)
flag_col = cfg.require("anomaly.flag_col")

display(summarize_anomalies_report(
    cfg=cfg,
    df=anomaly_df
))
display(
    anomaly_df.filter(pl.col(flag_col))
    .sort("resale_price_num", descending=True)
    .head(10)
)

In [ ]:
# Applies anomaly configured action: 'flag' keep all rows, 'quarantine' remove flagged rows out
cleaned_df, anomaly_quarantined_df = split_anomalies(
    cfg=cfg,
    df=anomaly_df
)

# Check whether anomalous rows to be quarantined are available
if anomaly_quarantined_df.height:
    quarantined_df = pl.concat([quarantined_df, anomaly_quarantined_df], how="diagonal_relaxed")

print("Cleaned: {cleaned_row_count} rows | Quarantined: {quarantined_row_count}".format(
    cleaned_row_count=cleaned_df.height,
    quarantined_row_count=quarantined_df.height
))

In [ ]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in profile_reference_dataset
report_paths = generate_ydata_profile(
    cfg=cfg,
    df=anomaly_df,
    profile_name="silver_post_anomaly_detection"
)
print(report_paths)

### 3.6 Write the cleaned and quarantined data outputs
Outputs will be written once after every silver layer step has concluded, so cleaned data includes the calculated remaining lease and anomaly columns whilst quarantined includes every rejected rows.
Additionally, the allowed values from the authoritative set are also saved as JSON for quick traceability.

In [ ]:
# Every combined row must be either cleaned or quarantined (i.e. nothing is dropped)
assert cleaned_df.height + quarantined_df.height == combined_df.height, "Rows at some point during Silver layer operations, check after combine step."

silver_paths = write_validation_outputs(
    cfg=cfg,
    valid_df=cleaned_df,
    quarantined_df=quarantined_df,
    authoritative_value_sets=authoritative_value_sets
)

print(silver_paths)

## 4. Gold Layer
### 4.1 Resale Identifier
Built from the validated data with the following structure: `S + BBB + PPP + YYMM + T`

| **Part** | **Rule**                                                                              | **Example**            |
|----------|---------------------------------------------------------------------------------------|------------------------|
| S        | Constant prefix                                                                       | S                      |
| BBB      | First 3 digits of `block` after removing non-digits, left-padded with 0               | Block `19` gives `019` |
| PPP      | First 3 digits of the average resale price of the (year-month, town, flat_type) group | `S$230,000` gives `230`    |
| YYMM     | year-month of the row                                                                 | `2012-01` gives `1201` |
| T        | First letter of the town                                                              | Ang Mo Kio gives `A`   |

Multiple rows may share the same identifier, therefore, uniqueness is enforced **without dropping any record as per requirements**. Rows that collide get a sequence suffix (-001, 002,...), ranked by a fixed tie-break order and then by every other column.
The result is deterministic and does not depend on the input row order.

In [ ]:
identifier_df = add_resale_identifier(
    cfg=cfg,
    validated_df=cleaned_df
)

# Summary of how many rows collide - i.e. need a suffix
display(pl.DataFrame([summarize_identifier_collisions(
    cfg=cfg,
    df=identifier_df
)]))

id_col = cfg.require("identifier.resale_id_col")
data_month_col = cfg.require("scope.data_month_col")
data_block_col = cfg.require("scope.data_block_col")
data_town_col = cfg.require("scope.data_town_col")
data_resale_price_col = cfg.require("scope.data_resale_price_col")

display(
    identifier_df.select(
        data_block_col,
        data_resale_price_col,
        data_month_col,
        data_town_col,
        id_col
    ).head(10)
)

In [ ]:
# Additional checks to ensure that 1) nothing was dropped 2) every identifier is unique 3) every identifier matches the format
# Note that many of these checks are built directly within the functions as well
assert identifier_df.height == cleaned_df.height
assert identifier_df[id_col].n_unique() == identifier_df.height
# Check format with regex - Note that this will fail if relevant configurations are changed in config.yaml
assert identifier_df[id_col].str.contains(r"^S\d{10}[A-Z](-\d{3,})?$").all(), "Identifier format has been violated"

transformed_identifier_path = write_transformed_identifier_df(
    cfg=cfg,
    transformed_df=identifier_df
)
print(transformed_identifier_path)

### 4.2 Deidentify the identifier from 4.1
The goal is to replace the identifier created in 3.1 with an **HMAC-SHA256** hash keyed with a secret (the pepper from `HASH_PEPPER`):
This method was determined as so to fulfill the requirements:
- **Be Irreversible**: HMAC-SHA256 is a one-way function
- **Preserve Uniqueness**: deterministic and collision-resistant, which means unique identifiers stay unique.
- **Protect against dictionary attacks**: since the identifier space is small and structured, a plain or publicly "salted" hash could be brute-forced by hashing every candidate (an attack form made even easier with AI Agent Swarms...). However, with this implementation, without the secret key, an attacker cannot do that.
    - Note: While yes, a random per-row salt would stop dictionary attacks too, it would also break the uniqueness requirement. Therefore, that wasn't used.

The secret is read from the environment and is never stored in the repository, the config or the notebook output.

In [ ]:
hashed_df = deidentify_identifier(
    cfg=cfg,
    transformed_df=identifier_df
)

hash_col = cfg.require("deidentify.hash_id_col")
display(hashed_df.select(
        data_block_col,
        data_resale_price_col,
        data_month_col,
        data_town_col,
        hash_col
).head(10))

In [ ]:
# Checks to ensure 1) same number of rows 2) every hash is unique 3) 64 hex characters 4) plain identifier is removed
assert hashed_df.height == identifier_df.height
assert hashed_df[hash_col].n_unique() == hashed_df.height
assert hashed_df[hash_col].str.contains(r"^[0-9a-f]{64}$").all()
assert id_col not in hashed_df.columns

hashed_path = write_hashed_output_df(
    cfg=cfg,
    hashed_df=hashed_df
)
print(hashed_path)

### 5. ETL Lifecycle profiling
To ensure evolving data insights have been analyzed throughout the ETL pipeline design and implementation lifecycle stages, the same profile was taken at every stage so the profiling report can describe how the data behaves from Bronze Layer to Gold Layer, including data in the quarantine bucket.
Additionally, row counts are reconciled to showcase that no data was lost.

In [ ]:
stage_profile = pl.concat([
    stage_profiles[0],
    profile_stage(df=cleaned_df, stage="silver_cleaned"),
    profile_stage(df=quarantined_df, stage="silver_quarantined"),
    profile_stage(df=identifier_df, stage="gold_transformed_identified"),
    profile_stage(df=hashed_df, stage="gold_hashed"),
])

# Add to profiling report folder
stage_profile.write_csv(cfg.path("profiling", create=True) / "stage_profile.csv")

display(
    stage_profile.select(
        "stage",
        "row_count"
    ).unique()
)

# Example: Show nulls and distinct values per column at each stage - columns that weren't present at a stage will be null
display(
    stage_profile.pivot(
        on="stage",
        index="column",
        values="distinct_count",
    )
)

### Deliverables Inventory
Short summary as to where to find and what makes up the five required output groups.

In [ ]:
groups = {
    "Raw": "bronze_raw",
    "Cleaned": "silver_cleaned",
    "Quarantined": "silver_quarantined",
    "Transformed": "gold_transformed",
    "Hashed": "gold_hashed"
}

inventory = []

for group, path_key in groups.items():
    for p in sorted(cfg.path(path_key).glob("*")):
        if p.is_file():
            if p.suffix == ".parquet":
                row_count = pl.scan_parquet(p).select(pl.len()).collect().item()
            elif p.suffix == ".csv":
                row_count = pl.scan_csv(p).select(pl.len()).collect().item()
            else:
                continue
            inventory.append({
                "Output group": group,
                "file": str(p.relative_to(REPO_ROOT)),
                "size_mb": round(p.stat().st_size / 1e6, 2),
                "rows": row_count,
            })

display(pl.DataFrame(inventory))